### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] !pip install sentence_transformers
# [PATCHED] !pip install faiss-cpu

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
working_folder = './ Drive/TransformersCode/03-restaurant/chatbot/'

In [ ]:
import os
import pandas as pd

csv_file_path = os.path.join(working_folder, 'indian_food.csv')
df = pd.read_csv(csv_file_path)

df.head()

In [ ]:
QA_file_path = working_folder + 'AR_QA.csv'

QAfaiss_file_path = working_folder + 'QA_faiss_index.bin'

In [ ]:
def create_qa_pairs(row):
    qa_pairs = []

    name = row['ARname']

    ingredients = row['ARingredients']

    diet = row['ARdiet']

    prep_time = row['prep_time']

    cook_time = row['cook_time']

    state = row['state']

    region = row['region']

    flavor_profile = row['ARflavor_profile']

    course = row['ARcourse']

    qa_pairs.append({"question": f"ما هي المكونات اللازمة لتحضير {name}؟", "answer": ingredients})
    qa_pairs.append({"question": f"ما هي المكونات المطلوبة لعمل {name}؟", "answer": ingredients})
    qa_pairs.append({"question": f"مما يتكون {name}؟", "answer": ingredients})

    qa_pairs.append({"question": f"ما هو نوع الحمية الغذائية ل {name}؟", "answer": diet})
    qa_pairs.append({"question": f"هل {name} نباتي أم غير نباتي؟", "answer": diet})
    qa_pairs.append({"question": f"ما هو النظام الغذائي ل {name}؟", "answer": diet})

    qa_pairs.append({"question": f"كم من الوقت يستغرق تحضير {name}؟", "answer": f"{prep_time} دقيقة"})
    qa_pairs.append({"question": f"ما هي مدة تحضير {name}؟", "answer": f"{prep_time} دقيقة"})
    qa_pairs.append({"question": f"كم دقيقة يستغرق تحضير {name}؟", "answer": f"{prep_time} دقيقة"})

    qa_pairs.append({"question": f"كم من الوقت يستغرق طهي {name}؟", "answer": f"{cook_time} دقيقة"})
    qa_pairs.append({"question": f"ما هي مدة طهي {name}؟", "answer": f"{cook_time} دقيقة"})
    qa_pairs.append({"question": f"كم دقيقة يستغرق طهي {name}؟", "answer": f"{cook_time} دقيقة"})

    qa_pairs.append({"question": f"ما هو أصل {name}؟", "answer": state})
    qa_pairs.append({"question": f"في أي ولاية تم اختراع {name}؟", "answer": state})
    qa_pairs.append({"question": f"إلى أي ولاية ينتمي {name}؟", "answer": state})

    qa_pairs.append({"question": f"في أي منطقة يتم تحضير {name}؟", "answer": region})
    qa_pairs.append({"question": f"ما هي المنطقة التي ينتمي إليها {name}؟", "answer": region})
    qa_pairs.append({"question": f"في أي منطقة تشتهر {name}؟", "answer": region})

    qa_pairs.append({"question": f"ما هو المذاق الخاص ب {name}؟", "answer": flavor_profile})
    qa_pairs.append({"question": f"كيف يكون طعم {name}؟", "answer": flavor_profile})
    qa_pairs.append({"question": f"ما هي نكهة {name}؟", "answer": flavor_profile})

    qa_pairs.append({"question": f"ما هو التصنيف الغذائي ل {name}؟", "answer": course})
    qa_pairs.append({"question": f"في أي وجبة يمكن تناول {name}؟", "answer": course})
    qa_pairs.append({"question": f"هل {name} يعد طبقاً رئيسياً أم جانبياً؟", "answer": course})

    return qa_pairs

In [ ]:
qa_list = df.apply(create_qa_pairs, axis=1).sum()

qa_df = pd.DataFrame(qa_list)

qa_df.to_csv(QA_file_path, index=False, encoding='UTF-8')

print("Q&A pairs CSV file generated successfully.")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v1')

def get_embeddings(texts):
    embeddings = model.encode(texts, convert_to_tensor=True)
    return embeddings.numpy()

questions = qa_df['question'].tolist()
question_embeddings = get_embeddings(questions)

In [ ]:
import faiss

index = faiss.IndexFlatIP(question_embeddings.shape[1])

faiss.normalize_L2(question_embeddings)

index.add(question_embeddings)

faiss.write_index(index, QAfaiss_file_path)